# Importing Libraries

In [ ]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
warnings.simplefilter("ignore")
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Reading data

In [ ]:
df = pd.read_csv('fraudTest.csv')
df.head()

In [ ]:
df.info()

### Data Cleaning

In [ ]:
df = df.drop('Unnamed: 0', axis = 1)

# check whether the data containing null values or not.
print('Sum of Missing Values:', df.isnull().sum().sum())

As you can see that all the columns are already clean.

### Data Visualization

1. Fradulent Trasactions

In [ ]:
d = df["is_fraud"].value_counts().reset_index()
d.columns = ['is_fraud', 'count']

# Create the pie chart with updated parameters
fig = px.pie(d, values="count", names=['No', 'Yes'], hole=0.40, opacity=0.9,
             labels={"is_fraud": "Fraud", "count": "Number of Samples"})
fig.update_layout(title=dict(text="Pie Chart of Fraudulent Transactions"))
fig.update_traces(textposition="outside", textinfo="percent+label")
fig.show()

2.   Gender Analysis of Fraudulant Trasactions

In [ ]:
df_fraud = df[df['is_fraud'] == 1]

plt.figure(figsize=(5, 5))
sns.set()
plt.title('Gender Analysis of Fraud Persons')
sns.countplot(x=df_fraud['gender'], color='tomato')
plt.xlabel('Gender')
plt.ylabel('Count')

1.   Category Analysis of Fraud persons




In [ ]:
plt.figure(figsize=(17,8))
sns.set()
plt.suptitle('Category Analysis of Fraud persons')
sns.countplot(df_fraud['category'], palette='winter')
plt.xlabel('Number of Trasactions')
plt.ylabel('Trasaction Category')

1. Distribution of Fruadulent Transaction Amount (K)



In [ ]:
amounts = df_fraud['amt']

bins = [0, 100, 500, 1000, 5000]
labels = ['0-100', '101-500', '501-1000', '1001-5000']

df_fraud['amount_range'] = pd.cut(amounts, bins=bins, labels=labels)

plt.figure(figsize=(8, 6))
sns.set(style="whitegrid")
sns.countplot(x='amount_range', data=df_fraud, palette='summer')
plt.title('Amount Distribution Across Different Ranges')
plt.xlabel('Amount Range')
plt.ylabel('Frequency')
plt.xticks()
plt.show()

### **Feature Selection**

In [ ]:
features = df[['category', 'amt', 'gender', 'city', 'state', 'job']]

# Target Selection
target = df['is_fraud']

In [ ]:
# Check whether the data is imbalanced or not.
target.value_counts()

It is clearly observed that the data is highly imbalanced. There are 1289169 records belonging to one class and only 7506 records are belonging to other class. In oprder to balance the data, we will use downsampling method.

### Downsampling

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
# Define the undersampling method
undersampler = RandomUnderSampler()

# Undersample the data
x_sampled, y_sampled = undersampler.fit_resample(features, target)

# Check the balanced data
y_sampled.value_counts()

It can clrealy seen that the data has been balanced now.

### Data Encoding

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Assuming df is your original DataFrame
categorical_columns = x_sampled.select_dtypes(include=['object']).columns

# Initialize OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, drop='first')

# Fit and transform the categorical columns
encoded_values = encoder.fit_transform(x_sampled[categorical_columns])

# Create a DataFrame for the encoded values
encoded_columns = encoder.get_feature_names_out(categorical_columns)
encoded_df = pd.DataFrame(encoded_values, columns=encoded_columns, index=x_sampled.index)

# Drop original categorical columns from df and concatenate the encoded columns
df_features = x_sampled.drop(columns=categorical_columns)
df_features = pd.concat([df_features, encoded_df], axis=1)

# Display the resulting DataFrame with transformed numerical columns
df_features.head()


As you can see that the data is encoded into numerical format.

In [ ]:
# Train-Test Splition
x_train, x_test, y_train, y_test = train_test_split(df_features, y_sampled,
                                                    test_size = 0.2,
                                                    random_state=42)

### Model Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(x_train, y_train)
y_predict = model.predict(x_test)
acc  = round(100 * accuracy_score(y_test, y_predict))
print(f'Accuracy of Model on Test data:{acc}%')

#### Accuracy

#### Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_predict)
print('\nConfusion-Matrix:\n',cm)
print('\n\nTotal records given:',len(x_test))
actual_fp = cm[0][0] + cm[0][1]
actual_hp = cm[1][0] + cm[1][1]
tp = cm[0][0]
tn =  cm[1][1]
print('\nActual Fraud person:',actual_fp)
print('True predictions of Fraud persons:',tp)
print('False predictions of Fraud persons:',cm[0][1])
print('Accuracy:',round(tp/actual_fp * 100), '%')
print('\nActual Honest persons:',actual_hp)
print('True predictions of Honest persons:',tn)
print('False predictions of Honest persons:',cm[1][0])
print('Accuracy:',round(tn/actual_hp * 100), '%')
print('\n\n')
plt.figure(figsize=(7,5))
sns.heatmap(confusion_matrix(y_test, y_predict),annot=True, fmt='g',cmap='viridis')
plt.tight_layout()
plt.show()

#### Classification Report

In [ ]:
print('\nClassification report :\n',classification_report(y_test, y_predict))

### Saving the Model

In [ ]:
import pickle

# Save the model to a file
with open('model.pkl', 'wb') as file:
    pickle.dump(model, file)

with open('encoder_model.pkl', 'wb') as file2:
    pickle.dump(encoder, file2)

